# ACM Research 2026 - Colab Setup Maia-3

Evaluate pretrained Maia-3 5M with S8 on the same player test sets used by Maia-2.

Run sections 1-3 every new session, then section 5 to measure accuracy. Section 4 contains the same one-time data preparation commands as the Maia-2 notebook. Use the existing frozen splits when comparing model results.

Select a T4 GPU for the full evaluation. Outputs are stored in Google Drive under `acm_chess/baselines_maia3/pgn/`.

## 1. Environment check

In [ ]:
!nvidia-smi
import sys
print(sys.version)

## 2. Mount Drive (persistent storage)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJ = '/content/drive/MyDrive/acm_chess'
!mkdir -p {PROJ}/data {PROJ}/checkpoints {PROJ}/runs
print(PROJ)

## 3. Install


In [ ]:
%cd /content
![ -d repo ] || git clone https://github.com/SchwiLynn/ACM_RESEARCH_2026-Training-a-chess-model.git repo
%cd /content/repo
!git pull -q
!pip install -q --ignore-requires-python -r requirements.txt
!pip install -q git+https://github.com/CSSLab/maia3.git

D = f'{PROJ}/data'
!mkdir -p {D}/raw {D}/moves {D}/splits
!ln -sfn {D}/raw data/raw
!ln -sfn {D}/moves data/moves
!ln -sfn {D}/splits data/splits

In [ ]:
from pathlib import Path
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
PLAYER = 'thakt'
print('Device:', DEVICE)

## 4. Get the data. DONT RERUN THIS

These are the same P5 data preparation cells used by the Maia-2 notebook. They download the player's PGN into `data/raw`, build CSVs in `data/moves`, and create the train/validation/test files in `data/splits`.

Run them once when preparing a new dataset. For an existing experiment, use the frozen test sets already in Drive. P5 rebuilds the CSVs and splits even when the PGN download is skipped.

In [ ]:
# DONT RERUN THIS. Only needed once: data is already in Drive after the first run.
# Test on one player first, then run everyone. Already-downloaded players are skipped.
!echo thakt > /tmp/one.txt
!python shared/p5_prepare_all_players.py /tmp/one.txt
# !python shared/p5_prepare_all_players.py config/eval_players.txt

In [ ]:
# Only needed once: data is already in Drive after the first run.
# Already-downloaded players are skipped.
# !echo thakt > /tmp/one.txt
# !python shared/p5_prepare_all_players.py /tmp/one.txt
!python shared/p5_prepare_all_players.py config/eval_players.txt

## 5. Baseline: base Maia-3 on every player's test set

S8 reads the same `data/splits/<player>/test.csv` as Maia-2, reconstructs history from `data/raw/<player>_rapid.pgn`, and uses P8 to compute top-1, top-3, confidence intervals, and per-phase accuracy.

The default Git clone is used, as in the Maia-2 notebook. S8 and `shared/evaluation_metrics.py` must be available in that checkout. Upload missing scripts to their matching folders under `/content/repo`.

First run a timing and accuracy check on 1,000 positions for `PLAYER`. The sample is saved separately under `baselines_maia3/pgn/sample_1000/`.

In [ ]:
for script in ('track2_finetunemaia3/s8_maia3_baseline.py',
               'shared/evaluation_metrics.py'):
    if not Path(script).is_file():
        raise FileNotFoundError(f'Upload {script} into /content/repo before running this section.')
!mkdir -p {PROJ}/baselines_maia3
!python track2_finetunemaia3/s8_maia3_baseline.py --data-root "{D}" --out-dir "{PROJ}/baselines_maia3" --maia2-dir "{PROJ}/baselines" --players {PLAYER} --limit 1000 --device {DEVICE}

Run the next cell to measure accuracy on all positions in every available player's test set. S8 prints the reports and saves prediction CSVs and JSON metrics into `acm_chess/baselines_maia3/pgn/`.

If Maia-2 baseline files are available in `acm_chess/baselines/`, S8 includes their scores in `comparison.csv`. Maia-3 accuracy is computed even when those files are missing.

In [ ]:
!python track2_finetunemaia3/s8_maia3_baseline.py --data-root "{D}" --out-dir "{PROJ}/baselines_maia3" --maia2-dir "{PROJ}/baselines" --device {DEVICE}

## 6. View the accuracy reports and table

Print each player's P8 report with top-1, top-3, confidence intervals, and top-1 by phase, matching the Maia-2 F3 report. The table below also shows opening, middlegame, and endgame accuracy as percentages. These results are read from the saved files.


In [ ]:
import json
import sys
from pathlib import Path
import pandas as pd
from IPython.display import display

sys.path.insert(0, '/content/repo/shared')
from evaluation_metrics import format_report

folder = Path(PROJ) / 'baselines_maia3' / 'pgn'
comparison = pd.read_csv(folder / 'comparison.csv')

for index, player in comparison['player'].items():
    saved = json.loads((folder / f'{player}.json').read_text())
    report = saved['evaluation']
    print(f'\n===== {player} =====')
    print(format_report(report))

    for phase in ('opening', 'middlegame', 'endgame'):
        comparison.loc[index, f'{phase}_top1'] = report['phases'][phase]['acc']

accuracy_columns = [
    'maia3_top1', 'maia3_top3',
    'opening_top1', 'middlegame_top1', 'endgame_top1',
]
accuracy_columns += [column for column in ('maia2_top1', 'maia2_top3')
                     if column in comparison.columns]

display(comparison.style.format(
    {column: '{:.2%}' for column in accuracy_columns}, na_rep='-'
))

## 7. Freeze the environment for the writeup

Run this after an evaluation used in the research log.

In [ ]:
!pip freeze > {PROJ}/runs/requirements_maia3.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv > {PROJ}/runs/gpu_maia3.txt
print('frozen')